In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt

import os, sys
os.chdir('..')

from sazz.samplers.AutomaticBoomerangSampler import AutomaticBoomerangSampler
from sazz.samplers.StickyAutomaticBoomerangSampler import StickyAutomaticBoomerangSampler

In [ ]:
from sazz.utils.bnn_utils import count_params, get_activation, find_reference_bnn
from sazz.models.bnn_torch import BNNGaussianPrior, BNNGaussianLikelihood
from sazz.models.models_torch import BayesianModel
from sazz.models.smoke_test import TorchTarget

from torch import Tensor
from typing import Optional, Sequence, Literal

def find_reference_bnn(
    energy_fn,
    D: int,
    dtype: torch.dtype = torch.float64,
    device: torch.device = torch.device("cpu"),
    n_steps: int = 4000,
    lr: float = 1e-2,
    model: Optional["BayesianModel"] = None,
    reference: Literal["prior", "fisher"] = "prior",
    layer_slices: Optional[Sequence[slice]] = None,
    n_samples_fisher: int = 100,
    per_layer_clip: bool = False,
) -> tuple[Tensor, Tensor]:
    """
    MAP via Adam + choice of reference precision for BNNs.

    Parameters
    ----------
    reference : {"prior", "fisher"}
        "prior"  — use the prior precision as Sigma_inv. Requires `model`
                   with a prior exposing precision_diag(). This is the
                   recommended default for BNNs: it gives orbits matched
                   to the prior scale and requires no tuning.
        "fisher" — empirical Fisher diagonal with per-layer clipping.
                   Historical behaviour, kept for reproducibility.
    """
    fn = model.energy if model is not None else energy_fn

    # --- MAP via Adam ---
    beta = torch.randn(D, dtype=dtype, device=device) * 0.01
    beta.requires_grad_(True)
    optimizer = torch.optim.Adam([beta], lr=lr)
    for _ in range(n_steps):
        optimizer.zero_grad()
        loss = fn(beta)
        loss.backward()
        optimizer.step()
    x_ref = beta.detach().clone()

    if reference == "prior":
        if model is None:
            raise ValueError("reference='prior' requires a model argument.")
        prec = model.prior.precision_diag().to(dtype=dtype, device=device)
        Sigma_inv = torch.diag(prec)

    elif reference == "fisher":
        grad_sq = torch.zeros(D, dtype=dtype, device=device)
        for _ in range(max(n_samples_fisher, 1)):
            b = x_ref.clone().requires_grad_(True)
            g, = torch.autograd.grad(fn(b), b)
            grad_sq += g ** 2
        grad_sq /= max(n_samples_fisher, 1)

        if layer_slices is None:
            layer_slices = [slice(0, D)]

        diag_prec = grad_sq.clone()
        if per_layer_clip:
            for sl in layer_slices:
                block = diag_prec[sl]
                if block.numel() == 0:
                    continue
                diag_prec[sl] = block.clamp(min=1.0, max=1e5)
        Sigma_inv = torch.diag(diag_prec)

    else:
        raise ValueError(f"Unknown reference type: {reference}")

    return x_ref, Sigma_inv


def make_bnn_regression(X, y, layer_sizes, activation="tanh",
                       prior_std_weight=1.0, prior_std_bias=1.0,
                       fan_in_scaling=True, noise_std=0.1,
                       adam_steps=1000, adam_lr=1e-2,
                       dtype=torch.float64, device="cpu"):
    X = X.to(dtype=dtype, device=device)
    y = y.to(dtype=dtype, device=device)
    D = count_params(layer_sizes)
    act = get_activation(activation)

    prior = BNNGaussianPrior(
        layer_sizes, prior_std_weight, prior_std_bias,
        fan_in_scaling, dtype, device,
    )
    likelihood = BNNGaussianLikelihood(X, y, layer_sizes, act, noise_std)
    model = BayesianModel(prior, likelihood)

    x_ref, Sigma_inv = find_reference_bnn(model.energy, D, 
                                          model=model, dtype=dtype,
                                          device=device,
                                          n_steps=adam_steps, lr=adam_lr)

    return TorchTarget(
        name=f"bnn_regression_{'x'.join(map(str, layer_sizes))}_{activation}",
        D=D,
        grad_target=model.grad_energy,
        x_ref=x_ref,
        Sigma_inv=Sigma_inv,
        meta={"model": model, "layer_sizes": layer_sizes},
    )

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

N_RESAMPLE = 50_000

# ── 1. Boston Housing ────────────────────────────────────────────
# 506 samples, 13 features, target = median home value (MEDV)
# sklearn removed load_boston; use the original CSV from StatLib
from sklearn.datasets import fetch_openml
boston_raw = fetch_openml(name="boston", version=1, as_frame=True, parser="auto")
X_boston = boston_raw.data.values.astype(float)
y_boston = boston_raw.target.values.astype(float)

# ── 2. Naval Propulsion Plants ───────────────────────────────────
# 11,934 samples, 16 features, target = GT compressor decay coeff (col 17)
df_naval = pd.read_csv("benchmarks_august/datasets/naval_data.txt",
                        sep=r"\s+", header=None)
X_naval = df_naval.iloc[:, :16].values.astype(float)
y_naval = df_naval.iloc[:, 16].values.astype(float)
n_naval = 1000 
rng = np.random.default_rng(42)
idx = rng.choice(len(df_naval), n_naval, replace=False)
X_naval, y_naval = X_naval[idx], y_naval[idx]

# ── 3. Energy Efficiency ─────────────────────────────────────────
# 768 samples, 8 features, target = Y1 (heating load)
df_energy = pd.read_excel("benchmarks_august/datasets/energy_data.xlsx")
X_energy = df_energy.iloc[:, :8].values.astype(float)
y_energy = df_energy.iloc[:, 8].values.astype(float)

# ── Standardize & split (Hernández-Lobato & Adams convention) ────
datasets = {}
for name, X, y in [("boston", X_boston, y_boston),
                    ("naval",  X_naval,  y_naval),
                    ("energy", X_energy, y_energy)
                    ]:
    # 90/10 train-test split
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    # Standardize using training statistics
    x_mean, x_std = X_tr.mean(axis=0), X_tr.std(axis=0)
    x_std[x_std == 0] = 1.0   # guard against constant columns (Naval has some)
    y_mean, y_std = y_tr.mean(), y_tr.std()

    X_tr = (X_tr - x_mean) / x_std
    X_te = (X_te - x_mean) / x_std
    y_tr = (y_tr - y_mean) / y_std
    y_te = (y_te - y_mean) / y_std

    datasets[name] = {
        "X_train": torch.tensor(X_tr, dtype=torch.float64),
        "y_train": torch.tensor(y_tr, dtype=torch.float64),
        "X_test":  torch.tensor(X_te, dtype=torch.float64),
        "y_test":  torch.tensor(y_te, dtype=torch.float64),
        # keep numpy versions for sklearn metrics if needed
        "x_mean": x_mean, "x_std": x_std,
        "y_mean": y_mean, "y_std": y_std,
    }
    print(f"{name:8s}  N={X.shape[0]:>6d}  D={X.shape[1]:>2d}  "
          f"train={X_tr.shape[0]}  test={X_te.shape[0]}")

In [ ]:

target_boston = make_bnn_regression(
    datasets['boston']['X_train'], datasets['boston']['y_train'],
    layer_sizes=[13, 50, 50, 1],
    activation="tanh",
    prior_std_weight=torch.sqrt(torch.tensor([2.0])),      # tight global weights
    prior_std_bias=1.0,        # loose biases
    fan_in_scaling=True,      # not scaling, since prior_std_weight=0.3 is already tight
    noise_std=0.5,             # appropriate noise assumption
)



In [ ]:
from sazz.utils.bnn_utils import make_kappa_vector_bnn
# Kappa per layer — middle layers sparser
kappa_boston = make_kappa_vector_bnn(
    target_boston.meta['layer_sizes'],
    #kappa_weights=[0.5, 0.05, 0.5],  # sparse middle layer
    kappa_weights=[5, 0.5, 5],
    kappa_biases=1e6,
)

sampler_boston_sticky = StickyAutomaticBoomerangSampler(
    grad_target=target_boston.grad_target,
    D=target_boston.D,
    thinning="pli",
    refresh_rate=1.0,
    kappa=kappa_boston
)

# Reference matches the prior — principled and needs no tuning
sampler_boston_sticky.preprocess(
    x_ref=target_boston.x_ref,
    Sigma_inv=target_boston.Sigma_inv
)

In [ ]:
N_SKELETON = 1_000
result_boston_sticky = sampler_boston_sticky.sample(N=N_SKELETON, diagnostics=True)